In [ ]:
import os

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Test") \
    .master("local[1]") \
    .getOrCreate()

path_file = r"C:/Users/rqz75/Documents/Cristina/Iron Hack/workspace/airflow/data/raw/ecommerce_orders_arrived.csv"

In [2]:
df = spark.read.csv(path_file, header=True,
    inferSchema=True)

In [3]:
df.show(5)
df.printSchema()

+--------+-----------------+-------+------+-------------+--------------------+
|order_id|    customer_name|product|amount|       region|           timestamp|
+--------+-----------------+-------+------+-------------+--------------------+
|       1|  Ashley Schaefer|  Shirt|567.82|North America|2026-05-31 12:46:...|
|       2| Antonio Phillips| Blouse|978.16|       Europe|2026-05-31 12:46:...|
|       3|     Angela Lynch| Blouse|406.42|       Africa|2026-05-31 12:46:...|
|       4|Cheryl Cunningham|  Jeans| 187.7|North America|2026-05-31 12:46:...|
|       5|   Laura Gonzalez|  Socks|629.81|      Oceania|2026-05-31 12:46:...|
+--------+-----------------+-------+------+-------------+--------------------+
only showing top 5 rows
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- region: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [4]:
#Change dataType
df_cast =( df.withColumn("customer_name",df.customer_name.cast("string"))
          .withColumn("product",df.product.cast("string"))
          .withColumn("amount",df.amount.cast("float"))
          )
df_cast.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: float (nullable = true)
 |-- region: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [5]:
#Fill null values
df_no_na = df_cast.na.fill({"customer_name": "unknown", "product": "unknown", "region": "unknown", "amount": 0})
df_no_na.show(5)

+--------+-----------------+-------+------+-------------+--------------------+
|order_id|    customer_name|product|amount|       region|           timestamp|
+--------+-----------------+-------+------+-------------+--------------------+
|       1|  Ashley Schaefer|  Shirt|567.82|North America|2026-05-31 12:46:...|
|       2| Antonio Phillips| Blouse|978.16|       Europe|2026-05-31 12:46:...|
|       3|     Angela Lynch| Blouse|406.42|       Africa|2026-05-31 12:46:...|
|       4|Cheryl Cunningham|  Jeans| 187.7|North America|2026-05-31 12:46:...|
|       5|   Laura Gonzalez|  Socks|629.81|      Oceania|2026-05-31 12:46:...|
+--------+-----------------+-------+------+-------------+--------------------+
only showing top 5 rows


In [8]:
import pandas
df_pandas = df_no_na.toPandas()

path_output = r"C:\Users\rqz75\Documents\Cristina\Iron Hack\workspace\airflow\data\processed\ecommerce_pipeline\ecommerce_orders_cleaned.csv"

df_pandas.to_csv(
    path_or_buf=path_output, 
    index=False, 
    header=True
)